# Week 3 — Context Engineering II

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-03-context-engineering-ii-content.html`.

**You will practice:**
1. Sliding-window truncation vs. rolling summarization on a synthetic conversation.
2. Generating embeddings and computing cosine similarity.
3. A minimal semantic search function (the seed of next week's RAG system).
4. A light ADK / LangChain preview of the summarization call.
5. Two open exercises.


In [1]:
%pip install -q --upgrade ollama langchain-ollama langgraph python-dotenv numpy pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import ollama
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()

# Inicializamos el cliente oficial de Ollama
client = ollama.Client()

# Definimos los nombres de los modelos locales
MODEL = "qwen2.5:14b"
EMBED_MODEL = "nomic-embed-text"

## 1. Sliding window vs. rolling summarization

In [1]:
history = [
    {"role": "user", "content": "Hi, I'm planning a trip to Peru."},
    {
        "role": "model",
        "content": (
            "Great! When are you thinking of traveling, and for how long?"
        ),
    },
    {"role": "user", "content": "Sometime in August, for about 2 weeks."},
    {
        "role": "model",
        "content": (
            "August is dry season in the Andes, good for Machu Picchu. Any"
            " budget in mind?"
        ),
    },
    {"role": "user", "content": "Around $2000 total, excluding flights."},
    {
        "role": "model",
        "content": (
            "That's workable for hostels and local transport. Interested in the"
            " Amazon too, or just the highlands?"
        ),
    },
    {"role": "user", "content": "Just the highlands. I don't do well with humidity."},
    {
        "role": "model",
        "content": (
            "Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it"
            " is."
        ),
    },
    {"role": "user", "content": "Also I'm vegetarian, will that be an issue?"},
    {
        "role": "model",
        "content": (
            "Not at all, Peruvian highland cuisine has plenty of vegetarian"
            " options."
        ),
    },
]


def sliding_window(history, max_turns=4):
  return history[-max_turns:]


print("--- Sliding window (last 4) ---")
for m in sliding_window(history, max_turns=4):
  print(m["role"], ":", m["content"])

--- Sliding window (last 4) ---
user : Just the highlands. I don't do well with humidity.
model : Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is.
user : Also I'm vegetarian, will that be an issue?
model : Not at all, Peruvian highland cuisine has plenty of vegetarian options.


In [3]:
import ollama

client = ollama.Client()
MODEL = "qwen2.5:14b"


def summarize_history(history):
  transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history)
  prompt = (
      "Summarize the key facts, decisions, and constraints from this"
      " conversation in 3-5 bullet points. Be specific.\n\n" + transcript
  )
  response = client.chat(
      model=MODEL,
      messages=[{"role": "user", "content": prompt}],
      options={"temperature": 0.0},
  )
  return response["message"]["content"]


def compact_context(history, keep_recent=4):
  if len(history) <= keep_recent:
    return history
  older, recent = history[:-keep_recent], history[-keep_recent:]
  summary = summarize_history(older)
  return [{
      "role": "system",
      "content": f"Conversation summary so far:\n{summary}",
  }] + recent


compacted = compact_context(history, keep_recent=4)
print("--- Rolling summarization + last 4 ---")
for m in compacted:
  print(m["role"], ":", m["content"])

--- Rolling summarization + last 4 ---
system : Conversation summary so far:
- Travel dates: Sometime in August for a duration of 2 weeks.
- Budget: $2000 total, excluding flight costs.
- Primary destination: Considering both the Andean highlands (including Machu Picchu) and potentially the Amazon region.
- Season: Traveling during the dry season in the Andes, which is favorable for visiting Machu Picchu.
- Accommodation and transport: Budget is suitable for hostels and local transportation.
user : Just the highlands. I don't do well with humidity.
model : Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is.
user : Also I'm vegetarian, will that be an issue?
model : Not at all, Peruvian highland cuisine has plenty of vegetarian options.


Compare the two outputs above. Sliding window silently drops "vegetarian" and "highlands only, no humidity"
if the window is small enough — summarization keeps them. Try lowering `max_turns` to 2 and see which strategy
still remembers the vegetarian constraint.

## 2. Embeddings and cosine similarity

In [9]:
import numpy as np
import ollama

client = ollama.Client()
EMBED_MODEL = "nomic-embed-text"


def embed(text):
  response = client.embed(model=EMBED_MODEL, input=text)
  return response.embeddings[0]


def cosine_similarity(a, b):
  a, b = np.array(a), np.array(b)
  return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


v_brake = embed("The bike's front brake feels loose.")
v_caliper = embed("The caliper on the front wheel isn't gripping properly.")
v_weather = embed("It might rain this weekend.")

print("brake vs caliper (related): ", round(cosine_similarity(v_brake, v_caliper), 3))
print("brake vs weather (unrelated):", round(cosine_similarity(v_brake, v_weather), 3))

brake vs caliper (related):  0.737
brake vs weather (unrelated): 0.463


## 3. A minimal semantic search function

In [10]:
candidates = [
    "To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.",
    "Flat tires are usually caused by punctures or worn-out inner tubes.",
    "Our rental bikes come in small, medium, and large frame sizes.",
    "Helmets are provided free of charge with every rental.",
    "The gear shifter may need cable tension adjustment if shifting feels sluggish.",
    "Rentals can be extended by messaging support at least 2 hours before the due time.",
]
candidate_vectors = [embed(c) for c in candidates]

def semantic_search(query, top_k=3):
    q_vec = embed(query)
    scored = [(cosine_similarity(q_vec, v), text) for v, text in zip(candidate_vectors, candidates)]
    scored.sort(reverse=True)
    return scored[:top_k]

for score, text in semantic_search("my brake feels wobbly and doesn't stop well"):
    print(f"{score:.3f}  {text}")

0.708  To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.
0.615  The gear shifter may need cable tension adjustment if shifting feels sluggish.
0.544  Flat tires are usually caused by punctures or worn-out inner tubes.


Notice the top result shares almost no exact words with the query ("wobbly", "doesn't stop well" vs.
"loose", "caliper bolts") — that's semantic search working as intended. Next week we scale this exact pattern
into a full RAG pipeline with chunking and a real vector database.

## 4. Light preview: the summarization call via ADK and LangChain

Same idea as previous weeks — reproducing one call (this time, `summarize_history`) through different tools.

In [11]:
import ollama

client = ollama.Client()
MODEL = "qwen2.5:14b"


class OllamaAgent:

  def __init__(self, model, name, instruction):
    self.model = model
    self.name = name
    self.instruction = instruction

  def run(self, prompt):
    response = client.chat(
        model=self.model,
        messages=[
            {"role": "system", "content": self.instruction},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0.0},
    )
    return response["message"]["content"]


summarizer_agent = OllamaAgent(
    model=MODEL,
    name="summarizer_agent",
    instruction=(
        "Summarize the key facts, decisions, and constraints from this"
        " conversation in 3-5 specific bullet points."
    ),
)

transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history[:-4])
print("ADK Agent summary:\n", summarizer_agent.run(transcript))

ADK Agent summary:
 - Travel dates: August, for a duration of 2 weeks.
- Budget: $2000 total, excluding airfare.
- Primary destination: Considering both the Andes (including Machu Picchu) and potentially the Amazon.
- Accommodation preference: Likely to stay in hostels based on the budget.
- Weather consideration: Traveling during the dry season in the Andes.


In [14]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

MODEL = "qwen2.5:14b"
EMBED_MODEL = "nomic-embed-text"

# 1. Chat Model Summarization
llm = ChatOllama(model=MODEL, temperature=0.0)
lc_summary = llm.invoke(
    "Summarize the key facts, decisions, and constraints from this conversation"
    " in 3-5 bullet points:\n\n"
    + transcript
)
print("LangChain summary:\n", lc_summary.content)

# 2. Embeddings
lc_embeddings = OllamaEmbeddings(model=EMBED_MODEL)
lc_vector = lc_embeddings.embed_query("The bike's front brake feels loose.")
print("\nLangChain embedding dims:", len(lc_vector))

LangChain summary:
 - Travel dates: August, for a duration of 2 weeks.
- Budget: $2000 total, excluding airfare.
- Primary destination: Considering both the Andean highlands (including Machu Picchu) and potentially the Amazon rainforest.

LangChain embedding dims: 768


## 5. Exercises

In [15]:
def semantic_search_filtered(query, top_k=3, threshold=0.5):
  q_vec = embed(query)
  scored = [
      (cosine_similarity(q_vec, v), text)
      for v, text in zip(candidate_vectors, candidates)
  ]

  # Sort descending by similarity score
  scored.sort(reverse=True, key=lambda x: x[0])

  # Filter by threshold
  filtered = [(score, text) for score, text in scored if score >= threshold]

  return filtered[:top_k]


# Test 1: Relevant query
print("=== Relevant Query Test ===")
for score, text in semantic_search_filtered(
    "my brake feels wobbly and doesn't stop well", threshold=0.5
):
  print(f"{score:.3f}  {text}")

# Test 2: Irrelevant query (should return no results)
print("\n=== Irrelevant Query Test ===")
results = semantic_search_filtered(
    "how do I code a binary search tree in Python?", threshold=0.5
)
print("Results found:", len(results))


=== Relevant Query Test ===
0.708  To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.
0.615  The gear shifter may need cable tension adjustment if shifting feels sluggish.
0.544  Flat tires are usually caused by punctures or worn-out inner tubes.

=== Irrelevant Query Test ===
Results found: 0


In [16]:
import json
import ollama
from pydantic import BaseModel, Field

client = ollama.Client()
MODEL = "qwen2.5:14b"


class TripMemory(BaseModel):
  destination: str = Field(description="Primary travel destination or country")
  month: str = Field(description="Month or season of travel")
  budget_usd: int = Field(description="Total budget in USD (excluding flights)")
  dietary_restrictions: list[str] = Field(
      default_factory=list, description="Dietary requirements or restrictions"
  )
  regions_of_interest: list[str] = Field(
      default_factory=list, description="Specific regions or places mentioned"
  )


transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history[:-4])

response = client.chat(
    model=MODEL,
    messages=[
      {
          "role": "system",
          "content": (
              "Extract the structured trip memory from the conversation in"
              " valid JSON matching this schema:"
              f" {TripMemory.model_json_schema()}"
          ),
      },
      {"role": "user", "content": transcript},
  ],
    format="json",  # Forces JSON output format from Ollama
    options={"temperature": 0.0},
)

# Parse JSON string into Pydantic model
raw_json = response["message"]["content"]
memory = TripMemory.model_validate_json(raw_json)

print("Extracted Trip Memory:\n", memory.model_dump_json(indent=2))

Extracted Trip Memory:
 {
  "destination": "Peru",
  "month": "August",
  "budget_usd": 2000,
  "dietary_restrictions": [],
  "regions_of_interest": [
    "Machu Picchu",
    "Amazon",
    "highlands"
  ]
}


## Next week

Week 4 — **Retrieval-Augmented Generation (RAG)**: chunking strategies, ChromaDB and FAISS, and building a
complete, functional RAG system — the foundation of your Corte 1 project. See `week-04-rag-content.html`.